# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets in the dataset, then inspect their fields and columns with their `@id`.

In [ ]:
# List all available record sets and their fields
print("Available Record Sets (with @id and name):\n")
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        meta = rs.to_json()
        print(f"- @id: {meta['@id']}, name: {meta.get('name', '')}")
        print("  Fields/Columns:")
        # print fields and columns as available
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                fmeta = f.to_json()
                print(f"    - Field @id: {fmeta['@id']}, name: {fmeta.get('name', '')}, dataType: {fmeta.get('dataType', '')}")
        if hasattr(rs, 'columns') and rs.columns:
            for c in rs.columns:
                cmeta = c.to_json()
                print(f"    - Column @id: {cmeta['@id']}, name: {cmeta.get('name', '')}, dataType: {cmeta.get('dataType', '')}")
        print()
else:
    print("No record_sets attribute found in metadata. Please inspect the metadata object directly.")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.

First, select a record set to explore (by `@id`). Example notebooks typically choose the main table (e.g. mass spectrometry proteins table).

In [ ]:
# Example: Collect all record sets' @id
record_sets = []
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        meta = rs.to_json()
        record_sets.append(meta['@id'])
else:
    raise ValueError("No record_sets found in metadata.")
print(f"Found record sets with @id:\n{record_sets}\n")

# Select the first available record set as main table (you may select differently based on printout above)
main_record_set_id = record_sets[0]

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set {record_set_id}")
    else:
        print(f"No records found for record set {record_set_id}")

# Show columns for main record set
if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we choose a numeric field and a grouping field based on the DataFrame columns list above.

> **Note:** All variables correspond to column names derived from the `@id` values.

In [ ]:
# Choose a numeric field and group field
df = dataframes[main_record_set_id]
# Print columns for reference
print(f"Available columns:\n{df.columns.tolist()}")

# Example: Suppose there is a numeric field '@id': 'cr:field/coverage', and group field '@id': 'cr:field/sample'
# Replace by appropriate @id from your actual field printout!
numeric_field = df.columns[1] if len(df.columns) > 1 else None  # heuristic: pick second column
group_field = None
for c in df.columns:
    if 'sample' in c.lower():
        group_field = c
        break

if numeric_field:
    threshold = df[numeric_field].mean()  # or use a fixed threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        display(grouped_df.head())
else:
    print("No suitable numeric_field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field and, if available, compare across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR² dataset from its Croissant JSON-LD schema.
- Explored the main record set structure and available numeric fields.
- Demonstrated filtering and normalization on numeric data, and visualized distributions.

For more detailed, domain-specific exploration, consult the dataset documentation and customize the above notebook sections for your analysis goals.